# Task 1: Data Modeling & Ledger Specification
## **Mana & Coins Game Architecture**
---
### **Guideline for Reviewers & Users**
This notebook serves as the authoritative technical specification and mock dataset generator for backend engineering and data platform teams.

**Key Highlights:**
1. **Zero-Touch Execution:** Click **Run All** (or run cells sequentially). All mathematical curves, sample ledger events, and auxiliary tables will be generated and saved automatically to the local folder.
2. **Single Append-Only Ledger:** Demonstrates how all currency mutations (`MANA`, `COIN`, `FIAT_USD`) are unified into one immutable table without `UPDATE` or `DELETE` statements.
3. **Zero-Trust Triangulation:** Implements strict idempotency across three event sources for real-money purchases: *Client Callback*, *S2S Payment Gateway Webhook*, and *Cron Reconciliation Audit*.

## 1. Mathematical Grounding & Formula Precedence

### Strict Mathematical Precedence Statement
> **Mathematical Assumption & Proof:**  
> In the economic formulas specified ($3 \times L^c$), the exponentiation operator strictly binds **only** to the level variable $L$ ($3 \times (L^c)$), and **never** to the leading coefficient.  
> - If the exponent applied to the coefficient $(3 \times L)^c$, at Level 1 with $c=1.1$, the action cost would evaluate to $(3 \times 1)^{1.1} = 3^{1.1} \approx 3.348$ (or $3^2 = 9$ if squared).  
> - Under strict operator precedence: $3 \times (1^{1.1}) = 3 \times 1 = 3.0$ Mana, matching the exact baseline stated in the game design rules.

Below are the analytical formulas implemented in pure Python for all economic mechanics:
- **Action Mana Cost ($m_a$):** $\text{round}(3 \times L^{1.1})$
- **Coins Per Dollar (CPD):** $4.32 \times m_a$
- **Mana Capacity ($C$):** $\text{round}((3 \times L_C^{1.1}) \times 30)$
- **Capacity Upgrade Cost:** $\text{round}(3 \times L_C^{2.2})$
- **Charging Speed (Mana/hr):** $\text{round}((3 \times L_{CS}^{1.1}) \times 1.5, 2)$
- **Speed Upgrade Cost:** $\text{round}(3 \times L_{CS}^{1.1})$

In [1]:
import json
import os
from dataclasses import dataclass, asdict
from datetime import datetime, timedelta
from typing import List, Optional, Dict, Any
import pandas as pd

class EconomyFormulas:
    """Encapsulates all exponential progression and inflation curves."""
    C_INFLATION: float = 1.1   # c = 1.1
    C2_CAPACITY: float = 30.0  # c2 = 30
    C3_CPD: float = 4.32       # c3 = 4.32
    C4_SPEED: float = 1.5      # c4 = 1.5

    @classmethod
    def action_mana_cost(cls, level: int) -> int:
        """m_a = round(3 * L ^ c)"""
        return round(3.0 * (level ** cls.C_INFLATION))

    @classmethod
    def capacity_limit(cls, level_capacity: int) -> int:
        """Capacity = (3 * L_C ^ c) * c2"""
        return round((3.0 * (level_capacity ** cls.C_INFLATION)) * cls.C2_CAPACITY)

    @classmethod
    def capacity_upgrade_cost(cls, level_capacity: int) -> int:
        """Cost = 3 * L_C ^ (2c) coins"""
        return round(3.0 * (level_capacity ** (2.0 * cls.C_INFLATION)))

    @classmethod
    def speed_mana_per_hour(cls, level_speed: int) -> float:
        """Mana/hour = (3 * L_CS ^ c) * c4"""
        return round((3.0 * (level_speed ** cls.C_INFLATION)) * cls.C4_SPEED, 2)

    @classmethod
    def speed_upgrade_cost(cls, level_speed: int) -> int:
        """Cost = 3 * L_CS ^ c coins"""
        return round(3.0 * (level_speed ** cls.C_INFLATION))

    @classmethod
    def coins_per_dollar(cls, level: int) -> float:
        """CPD = c3 * m_a"""
        m_a = cls.action_mana_cost(level)
        return round(cls.C3_CPD * m_a, 2)

    @classmethod
    def bundle_coins(cls, usd_price: float, level: int) -> int:
        """Coins awarded = usd_price * CPD"""
        return round(usd_price * cls.coins_per_dollar(level))

# Precedence Validation Proofs
print(f"-> Baseline Level 1: Action Cost = {EconomyFormulas.action_mana_cost(1)} Mana (Expected: 3)")
print(f"-> Baseline Level 1: Capacity    = {EconomyFormulas.capacity_limit(1)} Mana (Expected: 90)")
print(f"-> Mid-Game Level 5: Action Cost = {EconomyFormulas.action_mana_cost(5)} Mana | CPD = {EconomyFormulas.coins_per_dollar(5)}")
print(f"-> Late-Game Level 10: Action Cost = {EconomyFormulas.action_mana_cost(10)} Mana | CPD = {EconomyFormulas.coins_per_dollar(10)}")

-> Baseline Level 1: Action Cost = 3 Mana (Expected: 3)
-> Baseline Level 1: Capacity    = 90 Mana (Expected: 90)
-> Mid-Game Level 5: Action Cost = 18 Mana | CPD = 77.76
-> Late-Game Level 10: Action Cost = 38 Mana | CPD = 164.16


## 2. Immutable Ledger Event Emitter

The transaction table is modeled with strict event-sourcing principles:
- Every row represents an immutable event with a signed delta (`amount`).
- Running balances are derived dynamically from settled entries.
- Unverified client-side reports are isolated with `amount = 0`.

In [2]:
@dataclass
class TransactionRecord:
    transaction_id: str
    user_id: str
    created_at: str
    currency: str
    amount: float
    event_type: str
    user_level: int
    capacity_level: int
    speed_level: int
    source_system: str
    status: str
    external_transaction_id: Optional[str]
    idempotency_key: str
    metadata: str

class LedgerSimulator:
    """Simulates player lifecycles and produces append-only ledger events."""
    def __init__(self, start_time: datetime):
        self.current_time = start_time
        self.tx_counter = 1000
        self.records: List[TransactionRecord] = []
        self.balances: Dict[str, Dict[str, float]] = {}

    def _next_tx_id(self) -> str:
        self.tx_counter += 1
        return f"tx_{self.tx_counter}"

    def emit(
        self, user_id: str, minutes_offset: int, currency: str, amount: float,
        event_type: str, user_level: int, capacity_level: int, speed_level: int,
        source_system: str, status: str, idempotency_key: str,
        external_transaction_id: Optional[str] = None, metadata: Optional[Dict[str, Any]] = None
    ) -> TransactionRecord:
        self.current_time += timedelta(minutes=minutes_offset)
        user_bal = self.balances.setdefault(user_id, {"MANA": 0.0, "COIN": 0.0, "FIAT_USD": 0.0})
        
        if status == "SETTLED" and currency in user_bal:
            user_bal[currency] += amount

        meta = metadata or {}
        if status == "SETTLED" and currency in ("MANA", "COIN"):
            meta["balance_after"] = user_bal[currency]

        rec = TransactionRecord(
            transaction_id=self._next_tx_id(),
            user_id=user_id,
            created_at=self.current_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
            currency=currency,
            amount=amount,
            event_type=event_type,
            user_level=user_level,
            capacity_level=capacity_level,
            speed_level=speed_level,
            source_system=source_system,
            status=status,
            external_transaction_id=external_transaction_id,
            idempotency_key=idempotency_key,
            metadata=json.dumps(meta)
        )
        self.records.append(rec)
        return rec

## 3. Lifecycle Scenarios Execution (Data Contract Coverage)

The simulation below creates a deterministic, highly realistic dataset covering:
1. **Starter onboarding:** Welcome coin gift & background recharge up to capacity (90).
2. **Level 1 Actions:** Mana deductions at $m_a = 3$.
3. **Level 5 Progression & Payment Triangulation:**
   - Client claim: Recorded as `PENDING_VERIFICATION` with `amount = 0`.
   - S2S Webhook: Authoritative settlement granting 388 coins.
   - Fiat revenue booking: $4.99 in `FIAT_USD`.
   - Cron reconciliation: Deduplication check marked `DUPLICATE_IGNORED`.
4. **Economy Sinks:** Capacity upgrade, speed upgrade, 2x speed booster, and coin-to-mana conversion.
5. **Fraud Attempt:** Malicious client callback rejected with zero balance leakage.

In [3]:
sim = LedgerSimulator(start_time=datetime(2026, 10, 1, 8, 0, 0))
u1 = "usr_alex_01"
u2 = "usr_bella_02"

# 1. Level 1 Gameplay
sim.emit(u1, 0, "COIN", 25, "OCCASIONAL_GIFT", 1, 1, 1, "GAME_ENGINE", "SETTLED", "gift_welcome_alex_01", metadata={"note": "Starter welcome gift"})
sim.emit(u1, 10, "MANA", 90, "RECHARGE_TICK", 1, 1, 1, "GAME_ENGINE", "SETTLED", "recharge_u1_init_cap", metadata={"reason": "Auto-recharge to Level 1 capacity (90)"})
sim.emit(u1, 5, "MANA", -3, "ACTION_SPEND", 1, 1, 1, "GAME_ENGINE", "SETTLED", "act_u1_001", metadata={"action": "move_forward", "formula": "3 * 1^1.1 = 3"})
sim.emit(u1, 1, "MANA", -3, "ACTION_SPEND", 1, 1, 1, "GAME_ENGINE", "SETTLED", "act_u1_002", metadata={"action": "move_forward"})
sim.emit(u1, 64, "MANA", 6, "RECHARGE_TICK", 1, 1, 1, "GAME_ENGINE", "SETTLED", "recharge_u1_002", metadata={"reason": "Auto-recharge tops up back to capacity"})

# 2. Level 5 Gameplay & Payment Triangulation ($4.99 Bag -> 388 Coins)
sim.emit(u1, 160, "MANA", -18, "ACTION_SPEND", 5, 1, 1, "GAME_ENGINE", "SETTLED", "act_u1_010", metadata={"formula": "round(3 * 5^1.1) = 18"})
order_id = "GPA.3391-4820-9912"
# 2a. Client Callback (Untrusted -> amount = 0, PENDING)
sim.emit(u1, 5, "COIN", 0, "IAP_PURCHASE_CLAIM", 5, 1, 1, "CLIENT_CALLBACK", "PENDING_VERIFICATION", f"claim_{order_id}", external_transaction_id=order_id, metadata={"item_id": "coin_bag", "price_usd": 4.99})
# 2b. PG Webhook (Trusted -> amount = 388, SETTLED)
sim.emit(u1, 0, "COIN", 388, "IAP_PURCHASE", 5, 1, 1, "PG_WEBHOOK", "SETTLED", f"settle_{order_id}", external_transaction_id=order_id, metadata={"item_id": "coin_bag", "cpd": 77.76, "coins_awarded": 388})
# 2c. Fiat Revenue Booking
sim.emit(u1, 0, "FIAT_USD", 4.99, "IAP_REVENUE", 5, 1, 1, "PG_WEBHOOK", "SETTLED", f"fiat_{order_id}", external_transaction_id=order_id, metadata={"gateway": "Google_Play"})
# 2d. Cron Reconciliation Audit (Duplicate -> amount = 0)
sim.emit(u1, 15, "COIN", 0, "RECONCILIATION_AUDIT", 5, 1, 1, "RECON_JOB", "DUPLICATE_IGNORED", f"recon_{order_id}", external_transaction_id=order_id, metadata={"status": "ALREADY_SETTLED"})

# 3. Economy Sinks (Upgrades, Boosts, Exchanges)
sim.emit(u1, 5, "COIN", -14, "UPGRADE_CAPACITY", 5, 2, 1, "GAME_ENGINE", "SETTLED", "upg_cap_u1_lv2", metadata={"new_cap": EconomyFormulas.capacity_limit(2)})
sim.emit(u1, 5, "COIN", -6, "UPGRADE_SPEED", 5, 2, 2, "GAME_ENGINE", "SETTLED", "upg_spd_u1_lv2", metadata={"new_speed": EconomyFormulas.speed_mana_per_hour(2)})
sim.emit(u1, 30, "COIN", -20, "BUY_BOOST", 5, 2, 2, "GAME_ENGINE", "SETTLED", "boost_u1_001", metadata={"type": "2X_SPEED", "duration": 5})
sim.emit(u1, 5, "MANA", 2, "RECHARGE_TICK", 5, 2, 2, "GAME_ENGINE", "SETTLED", "recharge_boost_u1_001", metadata={"boost_active": True})
sim.emit(u1, 55, "COIN", -30, "COIN_TO_MANA_EXCHANGE", 5, 2, 2, "GAME_ENGINE", "SETTLED", "c2m_u1_coin", metadata={"mana_acquired": 100})
sim.emit(u1, 0, "MANA", 100, "COIN_TO_MANA_EXCHANGE", 5, 2, 2, "GAME_ENGINE", "SETTLED", "c2m_u1_mana", metadata={"coins_cost": 30})

# 4. Fraud Attempt Detection & Quarantine
ghost_order = "GPA.9999-0000-FAKE"
sim.emit(u2, 60, "COIN", 0, "IAP_PURCHASE_CLAIM", 10, 4, 3, "CLIENT_CALLBACK", "PENDING_VERIFICATION", f"claim_{ghost_order}", external_transaction_id=ghost_order, metadata={"item_id": "coin_barrel"})
sim.emit(u2, 15, "COIN", 0, "RECONCILIATION_AUDIT", 10, 4, 3, "RECON_JOB", "REJECTED", f"recon_{ghost_order}", external_transaction_id=ghost_order, metadata={"error": "PG_404_NOT_FOUND", "fraud_flag": True})

df_ledger = pd.DataFrame([asdict(r) for r in sim.records])
print(f" Successfully emitted {len(df_ledger)} ledger events across all operational scenarios.")

 Successfully emitted 18 ledger events across all operational scenarios.


## 4. Visual Inspection of the Single Append-Only Ledger
Below is the core ledger dataset demonstrating all transaction types, sources, and verification states.

In [4]:
cols_to_display = ["transaction_id", "user_id", "created_at", "currency", "amount", "event_type", "user_level", "source_system", "status", "external_transaction_id"]
display(df_ledger[cols_to_display])

,transaction_id,user_id,created_at,currency,amount,event_type,user_level,source_system,status,external_transaction_id
0,tx_1001,usr_alex_01,2026-10-01T08:00:00Z,COIN,25.00,OCCASIONAL_GIFT,1,GAME_ENGINE,SETTLED,NaN
1,tx_1002,usr_alex_01,2026-10-01T08:10:00Z,MANA,90.00,RECHARGE_TICK,1,GAME_ENGINE,SETTLED,NaN
2,tx_1003,usr_alex_01,2026-10-01T08:15:00Z,MANA,-3.00,ACTION_SPEND,1,GAME_ENGINE,SETTLED,NaN
3,tx_1004,usr_alex_01,2026-10-01T08:16:00Z,MANA,-3.00,ACTION_SPEND,1,GAME_ENGINE,SETTLED,NaN
4,tx_1005,usr_alex_01,2026-10-01T09:20:00Z,MANA,6.00,RECHARGE_TICK,1,GAME_ENGINE,SETTLED,NaN
5,tx_1006,usr_alex_01,2026-10-01T12:00:00Z,MANA,-18.00,ACTION_SPEND,5,GAME_ENGINE,SETTLED,NaN
6,tx_1007,usr_alex_01,2026-10-01T12:05:00Z,COIN,0.00,IAP_PURCHASE_CLAIM,5,CLIENT_CALLBACK,PENDING_VERIFICATION,GPA.3391-4820-9912
7,tx_1008,usr_alex_01,2026-10-01T12:05:00Z,COIN,388.00,IAP_PURCHASE,5,PG_WEBHOOK,SETTLED,GPA.3391-4820-9912
8,tx_1009,usr_alex_01,2026-10-01T12:05:00Z,FIAT_USD,4.99,IAP_REVENUE,5,PG_WEBHOOK,SETTLED,GPA.3391-4820-9912
9,tx_1010,usr_alex_01,2026-10-01T12:20:00Z,COIN,0.00,RECONCILIATION_AUDIT,5,RECON_JOB,DUPLICATE_IGNORED,GPA.3391-4820-9912


## 5. Auxiliary Dimensions & Progression Matrix (Levels 1 - 20)
These lookup matrices provide engineering and economy design teams with definitive scaling benchmarks.

In [5]:
# 1. Progression & Level Lookup Matrix (dim_game_level)
curve_rows = []
for l in range(1, 21):
    m_a = EconomyFormulas.action_mana_cost(l)
    curve_rows.append({
        "level": l,
        "action_mana_cost": m_a,
        "coins_per_dollar": EconomyFormulas.coins_per_dollar(l),
        "max_mana_capacity": EconomyFormulas.capacity_limit(l),
        "capacity_upgrade_cost": EconomyFormulas.capacity_upgrade_cost(l),
        "mana_recharge_rate_per_hour": EconomyFormulas.speed_mana_per_hour(l),
        "speed_upgrade_cost": EconomyFormulas.speed_upgrade_cost(l)
    })
df_game_level = pd.DataFrame(curve_rows)
display(df_game_level.head(10))

# 2. Store Items & Pricing Catalog (dim_shop_item)
df_shop_item = pd.DataFrame([
    {"item_id": "coin_drops", "item_name": "Drops", "price_usd": 0.99, "price_tier": 1},
    {"item_id": "coin_bag", "item_name": "Bag", "price_usd": 4.99, "price_tier": 2},
    {"item_id": "coin_chest", "item_name": "Chest", "price_usd": 9.99, "price_tier": 3},
    {"item_id": "coin_barrel", "item_name": "Barrel", "price_usd": 19.99, "price_tier": 4},
])

# 3. Player Balance & Progression Snapshot (dim_user)
df_user = pd.DataFrame([
    {
        "user_id": "usr_alex_01",
        "user_level": 5,
        "capacity_level": 2,
        "speed_level": 2,
        "max_mana_capacity": EconomyFormulas.capacity_limit(2),
        "mana_recharge_rate_per_hour": EconomyFormulas.speed_mana_per_hour(2),
        "mana_balance": 174,
        "coins_balance": 343,
        "lifetime_spend_usd": 4.99,
        "last_active_at": "2026-10-01T14:00:00Z"
    },
    {
        "user_id": "usr_bella_02",
        "user_level": 10,
        "capacity_level": 4,
        "speed_level": 3,
        "max_mana_capacity": EconomyFormulas.capacity_limit(4),
        "mana_recharge_rate_per_hour": EconomyFormulas.speed_mana_per_hour(3),
        "mana_balance": 320,
        "coins_balance": 150,
        "lifetime_spend_usd": 0.00,
        "last_active_at": "2026-10-01T15:15:00Z"
    }
])

,level,action_mana_cost,coins_per_dollar,max_mana_capacity,capacity_upgrade_cost,mana_recharge_rate_per_hour,speed_upgrade_cost
0,1,3,12.96,90,3,4.50,3
1,2,6,25.92,193,14,9.65,6
2,3,10,43.20,301,34,15.07,10
3,4,14,60.48,414,63,20.68,14
4,5,18,77.76,529,103,26.43,18
5,6,22,95.04,646,155,32.30,22
6,7,26,112.32,765,217,38.27,26
7,8,30,129.60,886,291,44.32,30
8,9,34,146.88,1009,377,50.45,34
9,10,38,164.16,1133,475,56.65,38


## 6. Zero-Touch Export to Production CSV Files
Saves all four artifacts directly into the directory so engineering teams have immediate access to raw CSV data.

In [ ]:
import os
target_dir = "release1_data_modeling" if os.path.exists("release1_data_modeling") else "."
os.makedirs(target_dir, exist_ok=True)

df_ledger.to_csv(os.path.join(target_dir, "transactions_sample.csv"), index=False)
df_game_level.to_csv(os.path.join(target_dir, "dim_game_level.csv"), index=False)
df_shop_item.to_csv(os.path.join(target_dir, "dim_shop_item.csv"), index=False)
df_user.to_csv(os.path.join(target_dir, "dim_user.csv"), index=False)

print(f" [SUCCESS] All four CSV files generated and saved to {target_dir}:")
print("  1. transactions_sample.csv")
print("  2. dim_shop_item.csv")
print("  3. dim_user.csv")
print("  4. dim_game_level.csv")
